# Appendix：ColabとPC・Google DriveのExcelファイル操作

このノートブックは **Google Colabで上から順に実行** します。Mac・Windows共通のコードです。

| やりたいこと | ファイルの移動 | 使う機能 |
|---|---|---|
| PCから読み込む | PC → Colabの `/content/` | `files.upload()` |
| PCへ保存する | Colabの `/content/` → PC | `files.download()` |
| Driveから読み込む | Google Drive → Colabのコード | `drive.mount()` + `pd.read_excel()` |
| Driveへ保存する | Colabのコード → Google Drive | `drive.mount()` + `to_excel()` |

`/content/` は一時領域です。Colabのセッションが終了すると、そこに作ったファイルは残りません。DriveのファイルはDriveに残ります。

**準備**：PC側の読み込みを試すにはExcelファイルが1つ必要です。手元になければ最初のセルで作った練習用ファイルをダウンロードして使えます。Google Driveの操作では本人の認証画面が表示されます。


## 0. 練習用Excelを作成する

まず、Colabの一時領域に練習用のExcelを作ります。PCにもDriveにも、この時点では保存されていません。


In [ ]:
from pathlib import Path
import pandas as pd

work = Path('/content')
work.mkdir(parents=True, exist_ok=True)
sample_path = work / 'appendix_sample_sales.xlsx'
sample = pd.DataFrame({
    '日付': ['2026-09-01', '2026-09-02', '2026-09-03'],
    '店舗': ['鹿児島', '霧島', '鹿児島'],
    '商品': ['茶', '菓子', '茶'],
    '売上金額': [1200, 800, 1500],
})
sample.to_excel(sample_path, index=False)
print('一時領域の練習用ファイル:', sample_path)
print(sample.head())


## 1. PC上のExcelファイルを読み込む

手元にExcelがなければ、**次のセルで練習用ExcelをPCへダウンロード**します。ブラウザのダウンロード先に保存されたファイルを、続くアップロード画面で選びます。MacもWindowsも同じコードです。


In [ ]:
from google.colab import files

files.download(str(sample_path))


次のセルを実行するとファイル選択画面が開きます。**Excelファイルを1つだけ選択**してください。アップロードしたファイルはColabの一時領域へコピーされます。


In [ ]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Excelファイルを1つだけ選んでください。')
local_filename = next(iter(uploaded))
if Path(local_filename).suffix.lower() != '.xlsx':
    raise ValueError('拡張子 .xlsx のExcelファイルを選んでください。')
local_input = work / local_filename
local_df = pd.read_excel(local_input)
print('PCからアップロードしたファイル:', local_input)
print(local_df.head())


## 2. 加工結果をPCへ保存する

ここでは「売上金額」列がある練習用Excelを想定します。実際のファイルを使う場合は列名を合わせてください。新しいExcelをColab上で作り、PCへダウンロードします。元のExcelは変更しません。


In [ ]:
if '売上金額' not in local_df.columns:
    raise ValueError('「売上金額」列がありません。自分のExcelの列名に合わせてください。')
local_result = local_df.copy()
local_result['売上金額'] = pd.to_numeric(local_result['売上金額'], errors='raise')
local_result['税込金額（10%の例）'] = (local_result['売上金額'] * 1.10).round().astype(int)
local_output = work / 'appendix_pc_result.xlsx'
local_result.to_excel(local_output, index=False)
print(local_result.head())
files.download(str(local_output))


## 3. Google Drive上のExcelファイルを読み込む

Drive接続時は自分のアカウントで認証します。以下は、最初に作った練習用Excelを **マイドライブ/ColabTips/Appendix** にコピーしてから読む例です。このフォルダには練習用ファイルだけを保存してください。


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_folder = Path('/content/drive/MyDrive/ColabTips/Appendix')
drive_folder.mkdir(parents=True, exist_ok=True)
drive_input = drive_folder / 'appendix_sample_sales.xlsx'
shutil.copy2(sample_path, drive_input)
print('Driveに作成した練習用ファイル:', drive_input)

drive_df = pd.read_excel(drive_input)
print('Driveから読み込んだ内容:')
print(drive_df.head())


**自分のExcelを読む場合**：`drive_input` に、自分のマイドライブ内のExcelのパスを指定します。例えば `Path('/content/drive/MyDrive/売上/2026年9月.xlsx')` です。Driveの画面に表示されるフォルダ名とパスが一致するか確認してください。


## 4. 加工結果をGoogle Driveへ保存する

読み込んだデータを加工し、Driveの同じ練習用フォルダに**別名のExcel**で保存します。元のExcelは変更しません。


In [ ]:
if '売上金額' not in drive_df.columns:
    raise ValueError('「売上金額」列がありません。自分のExcelの列名に合わせてください。')
drive_result = drive_df.copy()
drive_result['売上金額'] = pd.to_numeric(drive_result['売上金額'], errors='raise')
drive_result['税込金額（10%の例）'] = (drive_result['売上金額'] * 1.10).round().astype(int)
drive_output = drive_folder / 'appendix_drive_result.xlsx'
drive_result.to_excel(drive_output, index=False)
print('Google Driveに保存:', drive_output)
print(pd.read_excel(drive_output).head())


## 練習問題と注意

1. 自分で用意したExcelをPCからアップロードし、`head()` で中身を確認する。
2. 売上を店舗別に集計し、PCへダウンロードする。
3. Drive内の自分のファイルのパスを指定して読み込み、別名で保存する。

Excelに複数シート・数式・書式がある場合、`pd.read_excel()` で読み取った表を `to_excel()` すると元のブック全体を保持するわけではありません。元のExcelを直接上書きせず、まず別名で保存して内容を確認してください。
